# Qwen-2.5-0.5B SFT — OpenRA Intent Translation (Colab T4)

**Run order**:
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → Run all
3. Cell 2 弹窗时上传本地 `data/sft_train.jsonl` + `data/sft_val.jsonl`
4. 等 ~20 分钟训练完
5. 最后一个 cell 自动下载 `qwen05b_openra_lora.zip`

**Expected outcome**:
- training loss ~1.5 → 0.1 over 3 epochs
- val loss 跟训练 loss 不离谱 (差距 < 2x)
- 推理测试输出有效 JSON

## 1. 装依赖 (一次性装齐, ~3 分钟)

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

In [ ]:
# Verify everything imports — if this errors, fix install before going on
import torch, transformers, datasets, peft, trl, bitsandbytes, unsloth
print('torch:', torch.__version__, '/ cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('peft:', peft.__version__)
print('unsloth:', unsloth.__version__)

## 2. 上传训练数据

弹窗里同时选两个文件: `sft_train.jsonl` 和 `sft_val.jsonl` (在你本地 `D:\openra_mcp\data\` 下).

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
for name in uploaded:
    print(f'  {name}: {len(uploaded[name])} bytes')
assert 'sft_train.jsonl' in uploaded, 'missing sft_train.jsonl'
assert 'sft_val.jsonl' in uploaded, 'missing sft_val.jsonl'
print('OK')

## 3. 载 4-bit Qwen-2.5-0.5B-Instruct + 套 LoRA

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'trainable: {n_trainable:,}  ({100*n_trainable/n_total:.2f}% of {n_total:,})')

## 4. 载数据集 + apply chat_template

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={
    'train': 'sft_train.jsonl',
    'val':   'sft_val.jsonl',
})

def fmt(ex):
    return {
        'text': tokenizer.apply_chat_template(
            ex['messages'], tokenize=False, add_generation_prompt=False,
        )
    }

ds = ds.map(fmt, remove_columns=ds['train'].column_names)
print(f"train: {len(ds['train'])}  val: {len(ds['val'])}")
print('--- first train example (truncated) ---')
print(ds['train'][0]['text'][:600])
print('...')

## 5. 训练 (~15-25 分钟 on T4)

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir = 'qwen05b-lora',
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 2,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    warmup_ratio = 0.05,
    lr_scheduler_type = 'cosine',
    logging_steps = 10,
    save_strategy = 'epoch',
    eval_strategy = 'epoch',
    max_length = MAX_SEQ,
    bf16 = True,
    optim = 'adamw_8bit',
    report_to = 'none',
    seed = 42,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds['train'],
    eval_dataset = ds['val'],
    args = cfg,
    dataset_text_field = 'text',
)

result = trainer.train()
print(f"\n=== done: train_loss={result.training_loss:.4f} ===")

## 6. 快速推理测试 — 拿几句真实玩家话试

In [ ]:
import json, torch
FastLanguageModel.for_inference(model)

# 拿训练数据里的 system prompt (跟训练时一致)
with open('sft_train.jsonl', 'r', encoding='utf-8') as f:
    SYS = json.loads(f.readline())['messages'][0]['content']

SAMPLES = [
    '北队正面冲他重工',
    '残血的全撤回去',
    '南北夹击他老家',
    '切 alert',
    '总动员',
    '派几个去骚扰他经济',
    '看看场上情况',
    '全员守势',
    '坦克冲',
    '中路佯攻',
]

for nl in SAMPLES:
    msgs = [
        {'role': 'system', 'content': SYS},
        {'role': 'user',   'content': nl},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f'>> {nl}')
    print(f'   {gen}')
    print()

## 7. 保存 LoRA adapter + 下载 zip

In [ ]:
import shutil
from google.colab import files

model.save_pretrained('qwen05b_openra_lora')
tokenizer.save_pretrained('qwen05b_openra_lora')

shutil.make_archive('qwen05b_openra_lora', 'zip', 'qwen05b_openra_lora')
files.download('qwen05b_openra_lora.zip')
print('下载完成 — 解压到本地 D:\\openra_mcp\\outputs\\qwen05b-lora\\ 即可.')